# Building 3D & AR Experiences with the SceneView MCP Server

3D content is becoming a standard part of modern applications — from e-commerce product viewers to architectural walkthroughs to educational simulations. But generating correct 3D/AR code is hard for AI models: SDKs have threading constraints, nullable async loading, platform-specific gotchas, and API surfaces too large for any model to memorize from training data alone.

This cookbook shows how to solve that problem using the [Model Context Protocol (MCP)](https://modelcontextprotocol.io/). By connecting the [SceneView MCP server](https://www.npmjs.com/package/sceneview-mcp) to Claude, you give it expert-level knowledge of the [SceneView](https://github.com/sceneview/sceneview) 3D/AR SDK — including code generation, validation, and interactive 3D artifact creation.

The pattern demonstrated here — **packaging SDK expertise as MCP tools** — is applicable to any framework or SDK, not just SceneView.

## By the end of this cookbook, you will be able to:

1. Define MCP tool schemas and use them with the Anthropic Messages API
2. Implement a **"linter in the loop"** pattern where Claude validates its own generated code via tools before presenting it
3. Build an agentic tool-use loop (fetch reference → generate → validate → fix)
4. Understand how to package any SDK's expertise into MCP tools for Claude

## Prerequisites

**Required knowledge:**
- Python fundamentals — comfortable with functions, dicts, and basic API calls
- Basic understanding of [tool use with Claude](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/overview)

**Required tools:**
- Python 3.11 or higher
- An [Anthropic API key](https://console.anthropic.com/settings/keys)
- `anthropic` Python SDK

**Optional (for live MCP usage, no Python needed):**
- Node.js >= 18 for running the MCP server directly with Claude Desktop or Claude Code

### Quick install for Claude Desktop / Claude Code

If you want to use the MCP server directly (skip the notebook):

**Claude Desktop** — add to `~/Library/Application Support/Claude/claude_desktop_config.json`:
```json
{
  "mcpServers": {
    "sceneview": {
      "command": "npx",
      "args": ["-y", "sceneview-mcp"]
    }
  }
}
```

**Claude Code** — run:
```bash
claude mcp add sceneview -- npx -y sceneview-mcp
```

## Setup

In [ ]:
%pip install anthropic

Ensure your `.env` file contains:
```
ANTHROPIC_API_KEY=your_key_here
```

In [ ]:
import json
import os

import anthropic

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
MODEL = "claude-sonnet-4-6"

## Defining SceneView MCP tools

The SceneView MCP server exposes 12 tools covering code generation, validation, API reference, and interactive artifact creation. Below we define a representative subset as Anthropic API tool schemas — the same schemas the MCP server registers.

This lets us demonstrate the tool-use pattern with the Messages API directly, without needing to run the MCP server process. In production, Claude Desktop or Claude Code connects to the MCP server and these schemas are registered automatically.

| Tool | Purpose |
|------|---------|
| `get_sample` | Returns complete, compilable code for a given scenario |
| `validate_code` | Checks code for 15+ SDK-specific mistakes |
| `get_node_reference` | Returns full API reference for a node type |
| `create_3d_artifact` | Generates interactive HTML 3D content |
| `get_setup` | Returns Gradle/manifest setup instructions |

In [ ]:
SCENEVIEW_TOOLS = [
    {
        "name": "get_sample",
        "description": (
            "Returns a complete, compilable Kotlin or Swift sample for a given "
            "SceneView scenario. Scenarios cover 3D model viewing, AR placement, "
            "augmented images, physics, procedural geometry, and iOS equivalents."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "scenario": {
                    "type": "string",
                    "enum": [
                        "model-viewer",
                        "ar-model-viewer",
                        "ar-augmented-image",
                        "physics-demo",
                        "procedural-geometry",
                        "compose-ui-3d",
                        "ios-model-viewer",
                        "ios-ar-model-viewer",
                    ],
                    "description": "The scenario to fetch.",
                }
            },
            "required": ["scenario"],
        },
    },
    {
        "name": "validate_code",
        "description": (
            "Checks a Kotlin or Swift SceneView snippet for common mistakes: "
            "threading violations, wrong destroy order, missing null-checks, "
            "LightNode trailing-lambda bug, deprecated 2.x APIs. Always call "
            "this before presenting generated SceneView code to the user."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "code": {
                    "type": "string",
                    "description": "The Kotlin or Swift source code to validate.",
                }
            },
            "required": ["code"],
        },
    },
    {
        "name": "get_node_reference",
        "description": (
            "Returns the full API reference for a specific SceneView node type "
            "including parameters, types, defaults, and a usage example."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "nodeType": {
                    "type": "string",
                    "description": 'Node type to look up, e.g. "ModelNode", "LightNode", "ARScene".',
                }
            },
            "required": ["nodeType"],
        },
    },
    {
        "name": "create_3d_artifact",
        "description": (
            "Generates a self-contained HTML page with interactive 3D content. "
            'Types: "model-viewer" for 3D model viewing, "chart-3d" for 3D data '
            'visualization, "scene" for rich 3D scenes, "product-360" for product '
            "turntables with hotspot annotations."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "type": {
                    "type": "string",
                    "enum": ["model-viewer", "chart-3d", "scene", "product-360"],
                    "description": "The type of 3D artifact to generate.",
                },
                "modelUrl": {
                    "type": "string",
                    "description": "Public URL to a .glb model file (HTTPS, CORS-enabled).",
                },
                "title": {
                    "type": "string",
                    "description": "Title shown above the 3D viewer.",
                },
                "data": {
                    "type": "array",
                    "description": "Data points for chart-3d type. Array of {label, value, color} objects.",
                    "items": {
                        "type": "object",
                        "properties": {
                            "label": {"type": "string"},
                            "value": {"type": "number"},
                            "color": {"type": "string"},
                        },
                    },
                },
            },
            "required": ["type"],
        },
    },
    {
        "name": "get_setup",
        "description": "Returns the Gradle dependency and AndroidManifest snippet for SceneView.",
        "input_schema": {
            "type": "object",
            "properties": {
                "type": {
                    "type": "string",
                    "enum": ["3d", "ar"],
                    "description": '"3d" for 3D-only scenes. "ar" for augmented reality.',
                }
            },
            "required": ["type"],
        },
    },
]

## Example 1: Generating a 3D model viewer

Let's ask Claude to build a 3D model viewer. With the SceneView tools available, Claude will call `get_sample` to fetch the correct starting code rather than guessing at the API from training data.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=2048,
    tools=SCENEVIEW_TOOLS,
    messages=[
        {
            "role": "user",
            "content": (
                "Build me a Jetpack Compose screen that displays a 3D model "
                "of a chair with orbit controls. Use SceneView."
            ),
        }
    ],
)

print(f"Stop reason: {response.stop_reason}")
for block in response.content:
    if block.type == "tool_use":
        print(f"\nTool call: {block.name}")
        print(f"Input: {json.dumps(block.input, indent=2)}")
    elif block.type == "text":
        print(f"\nText: {block.text[:300]}")

Claude requests the `get_sample` tool with `scenario: "model-viewer"`. In a real MCP setup, the server would return the complete sample code. Let's simulate that response and continue the conversation to see how Claude uses it.

In [ ]:
# Simulate the MCP tool response with a realistic code sample.
# In production, the MCP server returns this automatically.
SAMPLE_MODEL_VIEWER = """@Composable
fun ModelViewerScreen() {
    val modelLoader = rememberModelLoader()
    val modelInstance = rememberModelInstance(modelLoader, "models/chair.glb")

    Scene(
        modifier = Modifier.fillMaxSize(),
        cameraManipulator = rememberCameraManipulator(
            CameraManipulator.Mode.ORBIT
        ),
        environment = rememberEnvironment(modelLoader, "environments/studio.hdr")
    ) {
        modelInstance?.let { instance ->
            ModelNode(
                modelInstance = instance,
                scaleToUnits = 1.0f
            )
        }
    }
}"""

# Find the tool_use block to get its ID
tool_use_block = next(b for b in response.content if b.type == "tool_use")

# Continue the conversation with the simulated tool result
followup = client.messages.create(
    model=MODEL,
    max_tokens=2048,
    tools=SCENEVIEW_TOOLS,
    messages=[
        {
            "role": "user",
            "content": (
                "Build me a Jetpack Compose screen that displays a 3D model "
                "of a chair with orbit controls. Use SceneView."
            ),
        },
        {"role": "assistant", "content": response.content},
        {
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": tool_use_block.id,
                    "content": SAMPLE_MODEL_VIEWER,
                }
            ],
        },
    ],
)

# Print Claude's final response
for block in followup.content:
    if block.type == "text":
        print(block.text)
    elif block.type == "tool_use":
        print(f"\n[Tool call: {block.name}({json.dumps(block.input)[:100]}...)]")

## The "linter in the loop" pattern

The most valuable tool in the SceneView MCP server is `validate_code`. It implements a pattern we call **"linter in the loop"**: Claude generates code, then validates it via a tool before presenting it to the user.

This catches bugs that are invisible to the model but catastrophic at runtime:

| Bug | What happens | How the validator catches it |
|-----|-------------|-----------------------------|
| `LightNode { ... }` (trailing lambda) | Compiles, but the light configuration is silently ignored | Pattern-matches `LightNode {` without `apply =` |
| Missing `?.let` on `rememberModelInstance()` | `NullPointerException` on first frame (model not loaded yet) | Checks that nullable return types are guarded |
| Calling `modelLoader.createModel()` on a background thread | JNI crash — Filament requires main thread | Detects `withContext(Dispatchers.IO)` around Filament calls |
| Using deprecated 2.x APIs (`ArSceneView`, `ArFrame`) | Compilation error on v3.x | Checks for known removed class/function names |

This pattern is applicable to **any** SDK: package the common mistakes into a validation tool, and Claude becomes self-correcting.

Let's see it in action with intentionally buggy code.

In [ ]:
BUGGY_CODE = """
@Composable
fun MyScene() {
    Scene(modifier = Modifier.fillMaxSize()) {
        // BUG: LightNode's apply is a named parameter, not a trailing lambda
        LightNode {
            intensity(100_000f)
        }
        ModelNode(
            modelInstance = rememberModelInstance(rememberModelLoader(), "model.glb")
        )
    }
}
"""

response_validate = client.messages.create(
    model=MODEL,
    max_tokens=2048,
    tools=SCENEVIEW_TOOLS,
    messages=[
        {
            "role": "user",
            "content": f"Check this SceneView code for bugs:\n```kotlin\n{BUGGY_CODE}\n```",
        }
    ],
)

for block in response_validate.content:
    if block.type == "tool_use":
        print(f"Tool: {block.name}")
        print("Claude sends the code to validate_code for automated checking.")
    elif block.type == "text":
        print(block.text)

The validator catches two issues:

1. **LightNode trailing lambda** — `LightNode { ... }` should be `LightNode(apply = { ... })`. The `apply` parameter is a named parameter, not a trailing lambda. Using the wrong syntax compiles but silently does nothing — the light has no effect.

2. **Missing null check** — `rememberModelInstance()` returns `ModelInstance?` (nullable). On the first composition, the model has not loaded yet, so the value is `null`. Passing it directly to `ModelNode` without a `?.let` guard causes a crash.

These are exactly the kind of SDK-specific bugs that generic AI models get wrong because the behavior is not obvious from the API signature alone. The MCP server encodes this domain knowledge as executable validation rules.

## Interactive 3D artifacts

The `create_3d_artifact` tool generates self-contained HTML pages with interactive 3D content. In Claude Desktop, these render inline as live artifacts the user can rotate, zoom, and interact with.

The artifacts use only CSS 3D transforms — no WebGL, no external dependencies, under 25 KB. This makes them work everywhere: Claude Desktop, browsers, mobile devices.

Let's ask Claude to visualize data in 3D.

In [ ]:
response_3d = client.messages.create(
    model=MODEL,
    max_tokens=2048,
    tools=SCENEVIEW_TOOLS,
    messages=[
        {
            "role": "user",
            "content": (
                "Show me a 3D bar chart of Q1 revenue by region: "
                "North America $4.2M, Europe $3.1M, Asia $2.8M, Latin America $1.5M."
            ),
        }
    ],
)

for block in response_3d.content:
    if block.type == "tool_use":
        print(f"Tool: {block.name}")
        print(f"Input:\n{json.dumps(block.input, indent=2)}")
    elif block.type == "text":
        print(block.text)

Claude calls `create_3d_artifact` with `type: "chart-3d"` and the structured data points. The MCP server generates a complete HTML page using CSS 3D transforms that renders an interactive 3D bar chart.

In Claude Desktop, this HTML is rendered inline as a live artifact — the user sees a 3D chart they can rotate by dragging. No WebGL dependencies, no external scripts, works in any browser.

## Agentic loop: generate-validate-fix cycle

In practice, Claude uses the tools in an agentic loop:

1. **Fetch** — `get_sample` or `get_node_reference` for the right starting code or API docs
2. **Generate** — produce code based on the user's request
3. **Validate** — `validate_code` checks for SDK-specific mistakes
4. **Fix** — correct any issues found
5. **Present** — return validated code to the user

Here is a helper that runs this loop, processing tool calls until Claude produces a final text response. In a real MCP integration, the MCP server handles the tool execution; here we use mock responses for demonstration.

In [ ]:
def run_sceneview_agent(user_prompt: str, tool_results: dict | None = None) -> str:
    """Run a multi-turn tool-use loop with SceneView tools.

    Args:
        user_prompt: The user's request.
        tool_results: Optional dict mapping tool names to mock responses
                      for demonstration without the live MCP server.

    Returns:
        Claude's final text response after all tool calls resolve.
    """
    tool_results = tool_results or {}
    messages = [{"role": "user", "content": user_prompt}]

    for turn in range(5):  # cap at 5 tool-use turns to avoid runaway loops
        response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            tools=SCENEVIEW_TOOLS,
            messages=messages,
        )

        # If Claude is done (no more tool calls), return the text
        if response.stop_reason == "end_turn":
            return "\n".join(
                block.text for block in response.content if block.type == "text"
            )

        # Process tool calls
        messages.append({"role": "assistant", "content": response.content})

        tool_result_contents = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"  Turn {turn + 1} -> {block.name}({list(block.input.keys())})")
                # Use mock result or a default acknowledgment
                result = tool_results.get(
                    block.name,
                    f"Tool {block.name} executed successfully. (Mock response for demo)",
                )
                tool_result_contents.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    }
                )

        messages.append({"role": "user", "content": tool_result_contents})

    return "Max tool-use turns reached."


print("Agent helper defined.")

In [ ]:
# Demonstrate the agentic loop with a mock tool response
result = run_sceneview_agent(
    "What Gradle setup do I need for a SceneView AR project?",
    tool_results={
        "get_setup": (
            "// build.gradle.kts (Module: app)\n"
            "dependencies {\n"
            '    implementation("io.github.sceneview:arsceneview:3.3.0")\n'
            "}\n\n"
            "// AndroidManifest.xml\n"
            '<uses-permission android:name="android.permission.CAMERA" />\n'
            '<uses-feature android:name="android.hardware.camera.ar" '
            'android:required="true" />\n'
            "<application>\n"
            '    <meta-data android:name="com.google.ar.core" '
            'android:value="required" />\n'
            "</application>"
        ),
    },
)

print("\n--- Claude's response ---")
print(result[:600])

## Real-world use cases

The SceneView MCP server enables Claude to handle a wide range of 3D/AR development tasks:

| Use case | Example prompt | Tools used |
|----------|---------------|------------|
| **E-commerce** | "Add a 3D product viewer to my app" | `get_sample` -> `validate_code` |
| **Architecture** | "Build an AR app that places furniture in a room" | `get_setup(ar)` -> `get_sample(ar-model-viewer)` |
| **Data visualization** | "Show Q1 revenue as an interactive 3D chart" | `create_3d_artifact(chart-3d)` |
| **Education** | "Create an interactive 3D anatomy model viewer" | `create_3d_artifact(model-viewer)` |
| **iOS development** | "Set up SceneView for iOS with SwiftUI" | `get_ios_setup` -> `get_sample(ios-model-viewer)` |
| **API lookup** | "What parameters does LightNode accept?" | `get_node_reference(LightNode)` |
| **Migration** | "Migrate my SceneView 2.x code to 3.0" | `get_migration_guide` |
| **Debugging** | "Check my SceneView code for bugs" | `validate_code` |

The MCP server handles all the domain expertise — threading rules, Filament JNI constraints, nullable model loading, platform-specific gotchas — so Claude generates correct code on the first try.

## Key takeaways

### 1. Domain-specific tools beat general-purpose training

Claude's training data includes SceneView, but it cannot keep up with API changes, SDK version differences, or subtle runtime behaviors. MCP tools provide **current, authoritative, executable knowledge** that the model can query at inference time.

### 2. The "linter in the loop" pattern is broadly applicable

The `validate_code` tool demonstrates a powerful pattern: let Claude generate code freely, then validate it via a tool before presenting it. This catches bugs that even expert developers miss — silent misconfigurations, threading violations, nullable type hazards. You can implement this pattern for any SDK by encoding its common pitfalls into a validation tool.

### 3. Interactive artifacts extend Claude beyond text

The `create_3d_artifact` tool generates self-contained HTML with interactive 3D content using only CSS 3D transforms — no WebGL, under 25 KB. This demonstrates how MCP tools can produce rich visual outputs, not just code or text.

### 4. The agentic loop ensures correctness

The fetch -> generate -> validate -> fix cycle means Claude's output improves with each tool call. This is more reliable than a single-shot generation, especially for complex SDK code.

## Build your own SDK MCP server

The pattern demonstrated here works for **any** SDK or framework. To build your own:

1. **`get_sample`** — Curate 5-10 complete, working code samples covering common scenarios. These serve as the ground truth Claude starts from.

2. **`validate_code`** — Catalog the top 10-20 mistakes developers make with your SDK. Encode each as a pattern-matching rule. This is the highest-value tool — it makes Claude self-correcting.

3. **`get_reference`** — Expose your API reference as a queryable tool. This handles the long tail of questions that samples don't cover.

4. **`create_artifact`** (optional) — If your SDK has visual output, generate interactive previews as HTML artifacts.

The SceneView MCP server is [open source](https://github.com/sceneview/sceneview/tree/main/mcp) and can serve as a template.

## Next steps

- **Try it live**: Install the MCP server in [Claude Desktop or Claude Code](#prerequisites) and ask Claude to build a 3D app
- **Explore the source**: [SceneView MCP server on GitHub](https://github.com/sceneview/sceneview/tree/main/mcp)
- **Build your own**: Use the pattern above to create an MCP server for your SDK
- **Learn more about MCP**: [Model Context Protocol documentation](https://modelcontextprotocol.io/)
- **Learn more about tool use**: [Anthropic tool use guide](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/overview)

### Resources

| Resource | Link |
|----------|------|
| SceneView GitHub | [github.com/sceneview/sceneview](https://github.com/sceneview/sceneview) |
| SceneView MCP on npm | [npmjs.com/package/sceneview-mcp](https://www.npmjs.com/package/sceneview-mcp) |
| Model Context Protocol | [modelcontextprotocol.io](https://modelcontextprotocol.io/) |
| Anthropic tool use guide | [docs.anthropic.com](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/overview) |